# Chapter 8: Type Hints in Functions
*From: Fluent Python by Luciano Ramalho*

This notebook distills the key concepts of Python's gradual type system as applied to function signatures. You will learn how type hints work with Mypy, explore the major annotation types (`Any`, `Optional`, `Union`, generics, `TypeVar`, `Protocol`, `Callable`), and practice writing well-typed Python.

**Related wiki articles:** [[gradual-typing-and-mypy]], [[any-optional-union]], [[generic-collections-and-tuples]], [[abcs-in-type-hints]], [[typevar-and-parameterized-generics]], [[static-protocols]], [[callable-and-noreturn]]

## TL;DR

- Python uses a **gradual type system**: type hints are always optional, ignored at runtime, and checked by tools like Mypy.
- **`Any`** is the escape hatch -- consistent-with every type in both directions. **`object`** is the universal superclass but has a narrow interface.
- **`Optional[X]`** means `X | None`. **`Union[X, Y]`** means either `X` or `Y`. Python 3.10+ allows `X | Y` syntax.
- **Generic collections** like `list[str]`, `dict[str, int]` constrain element types. Tuples have three forms: records, named tuples, variable-length.
- **ABCs** (`Mapping`, `Sequence`, `Iterable`) as parameter types follow Postel's law: be liberal in what you accept.
- **`TypeVar`** creates type variables to express "return type matches input type". Can be restricted or bounded.
- **`Protocol`** enables static duck typing: define structural interfaces checked at type-check time without inheritance.
- **`Callable[[ParamTypes], ReturnType]`** annotates callbacks. **`NoReturn`** marks functions that always raise.

---
## 1. Gradual Typing and Mypy

Python's type system is **gradual**: hints are optional, not enforced at runtime, and don't affect performance. Mypy (or Pyright, Pyre, pytype) reads them statically.

Key properties of gradual typing:
- **Optional**: unannotated code is valid; the checker assumes `Any`.
- **Not enforced at runtime**: hints are metadata, not contracts.
- **No performance boost**: the runtime ignores them (unlike, say, Cython).

See also: [[gradual-typing-and-mypy]]

In [ ]:
# A simple function -- no hints, Mypy ignores it by default
def show_count(count, word):
    if count == 1:
        return f"1 {word}"
    count_str = str(count) if count else "no"
    return f"{count_str} {word}s"

# Works fine at runtime
print(show_count(99, "bird"))   # 99 birds
print(show_count(1, "bird"))    # 1 bird
print(show_count(0, "bird"))    # no birds

Now let's add type hints incrementally -- the gradual typing way:

In [ ]:
# Step 1: Add just the return type (Mypy will then inspect arguments)
def show_count_v2(count, word) -> str:
    if count == 1:
        return f"1 {word}"
    count_str = str(count) if count else "no"
    return f"{count_str} {word}s"

# Step 2: Fully annotated
def show_count_v3(count: int, word: str) -> str:
    if count == 1:
        return f"1 {word}"
    count_str = str(count) if count else "no"
    return f"{count_str} {word}s"

# Step 3: With optional plural parameter
def show_count_v4(count: int, singular: str, plural: str = "") -> str:
    if count == 1:
        return f"1 {singular}"
    count_str = str(count) if count else "no"
    if not plural:
        plural = singular + "s"
    return f"{count_str} {plural}"

print(show_count_v4(3, "mouse", "mice"))  # 3 mice
print(show_count_v4(2, "bird"))            # 2 birds

### Duck Typing vs Nominal Typing

- **Duck typing** (runtime): if it quacks, it's a duck. No declared types on variables.
- **Nominal typing** (static): the declared type matters. `Duck` is a subclass of `Bird`, so a `Duck` can go where a `Bird` is expected -- but not vice versa.

In [ ]:
class Bird:
    pass

class Duck(Bird):
    def quack(self):
        print("Quack!")

def alert(birdie):
    "No type hints -- Mypy ignores this."
    birdie.quack()

def alert_duck(birdie: Duck) -> None:
    "Accepts Duck -- Mypy is happy."
    birdie.quack()

def alert_bird(birdie: Bird) -> None:
    "Accepts Bird -- Mypy flags birdie.quack() because Bird has no quack!"
    birdie.quack()  # type: ignore  # Mypy error, but works at runtime for Duck

daffy = Duck()
alert(daffy)       # Quack! -- duck typing
alert_duck(daffy)  # Quack! -- nominal typing OK
alert_bird(daffy)  # Quack! -- works at runtime, but Mypy would flag the function body

---
## 2. Any, Optional, and Union Types

See also: [[any-optional-union]]

### The Any Type

`Any` is the wildcard of gradual typing. An unannotated parameter is implicitly `Any`. It is **consistent-with every type** in both directions (unlike `object`, which only accepts everything but has a narrow interface).

In [ ]:
from typing import Any

# These two signatures are equivalent:
def double_implicit(x):
    return x * 2

def double_any(x: Any) -> Any:
    return x * 2

# Both accept anything and Mypy won't complain about x * 2

# But this would FAIL type checking:
def double_object(x: object) -> object:
    return x * 2  # type: ignore  # Mypy error: object doesn't support __mul__

# The difference: Any supports every operation; object has a minimal interface
print(double_implicit(4))        # 8
print(double_implicit("ha"))     # haha
print(double_implicit([1, 2]))   # [1, 2, 1, 2]

### consistent-with Rules

1. If `T2` is a subtype of `T1`, then `T2` is consistent-with `T1` (Liskov Substitution).
2. Every type is consistent-with `Any` (you can pass anything to an `Any` parameter).
3. `Any` is consistent-with every type (a value of type `Any` can go anywhere).

### Optional and Union

In [ ]:
from typing import Optional, Union

# Optional[X] means X or None -- it's shorthand for Union[X, None]
def show_count_opt(count: int, singular: str, plural: Optional[str] = None) -> str:
    if count == 1:
        return f"1 {singular}"
    count_str = str(count) if count else "no"
    if not plural:
        plural = singular + "s"
    return f"{count_str} {plural}"

print(show_count_opt(3, "mouse", "mice"))  # 3 mice
print(show_count_opt(2, "child"))           # 2 childs -- oops, but type-correct!

# Python 3.10+ syntax: X | Y instead of Union[X, Y]
def parse_token(token: str) -> str | float:
    try:
        return float(token)
    except ValueError:
        return token

print(parse_token("3.14"))    # 3.14 (float)
print(parse_token("hello"))   # hello (str)
print(type(parse_token("3.14")))  # <class 'float'>

> **Tip:** `Optional` is a confusing name -- it does NOT make the parameter optional.
> What makes a parameter optional is the default value (`= None`).
> `Optional[str]` just means "the type is `str | None`".

---
## 3. Generic Collections and Tuple Types

See also: [[generic-collections-and-tuples]]

Generic type hints constrain element types in collections. Since Python 3.9, you can use built-in collection types directly (`list[str]`, `dict[str, int]`).

In [ ]:
# Generic list
def tokenize(text: str) -> list[str]:
    return text.upper().split()

print(tokenize("the quick brown fox"))

# Generic dict -- inverted index example
import unicodedata
import re

RE_WORD = re.compile(r"\w+")

def name_index(start: int = 32, end: int = 128) -> dict[str, set[str]]:
    index: dict[str, set[str]] = {}
    for char in (chr(i) for i in range(start, end)):
        if name := unicodedata.name(char, ""):
            for word in RE_WORD.findall(name.upper()):
                index.setdefault(word, set()).add(char)
    return index

idx = name_index(32, 65)
print("SIGN:", idx.get("SIGN", set()))
print("DIGIT:", idx.get("DIGIT", set()))

### Tuple Types -- Three Forms

| Form | Meaning | Example |
|------|---------|---------|
| `tuple[X, Y]` | Fixed record with X then Y | `tuple[str, float]` |
| `NamedTuple` | Record with named fields | `Coordinate(lat, lon)` |
| `tuple[X, ...]` | Variable-length, all same type | `tuple[int, ...]` |

In [ ]:
from typing import NamedTuple

# 1. Tuple as record (fixed fields)
def geo_display(lat_lon: tuple[float, float]) -> str:
    lat, lon = lat_lon
    ns = "N" if lat >= 0 else "S"
    ew = "E" if lon >= 0 else "W"
    return f"{abs(lat):0.1f} deg {ns}, {abs(lon):0.1f} deg {ew}"

print(geo_display((31.23, 121.47)))  # Shanghai

# 2. Named tuple
class Coordinate(NamedTuple):
    lat: float
    lon: float

shanghai = Coordinate(31.23, 121.47)
print(geo_display(shanghai))  # Also works! NamedTuple is consistent-with tuple

# 3. Variable-length tuple
def columnize(sequence: list[str], num_columns: int = 0) -> list[tuple[str, ...]]:
    if num_columns == 0:
        num_columns = round(len(sequence) ** 0.5)
    num_rows, remainder = divmod(len(sequence), num_columns)
    num_rows += bool(remainder)
    return [tuple(sequence[i::num_rows]) for i in range(num_rows)]

animals = "drake fawn heron ibex koala lynx tahr xerus yak zapus".split()
for row in columnize(animals):
    print("".join(f"{word:10}" for word in row))

---
## 4. Abstract Base Classes in Type Hints

See also: [[abcs-in-type-hints]]

**Postel's Law**: Be conservative in what you send, be liberal in what you accept.

- **Parameters**: use ABCs like `Mapping`, `Sequence`, `Iterable` (liberal)
- **Return types**: use concrete types like `list`, `dict` (conservative)

In [ ]:
from collections.abc import Mapping, Sequence, Iterable

# GOOD: accepts any Mapping (dict, defaultdict, ChainMap, UserDict subclass...)
def name2hex(name: str, color_map: Mapping[str, int]) -> str:
    code = color_map.get(name.lower(), 0x000000)
    return f"#{code:06x}"

# Works with dict
print(name2hex("red", {"red": 0xFF0000, "green": 0x00FF00}))

# Works with any Mapping subtype
from collections import OrderedDict
od = OrderedDict(red=0xFF0000, blue=0x0000FF)
print(name2hex("blue", od))

# Iterable for maximum flexibility
FromTo = tuple[str, str]

def zip_replace(text: str, changes: Iterable[FromTo]) -> str:
    for from_, to in changes:
        text = text.replace(from_, to)
    return text

l33t = [("a", "4"), ("e", "3"), ("i", "1"), ("o", "0")]
print(zip_replace("mad skilled noob powned leet", l33t))

> **Why ABCs over concrete types?** If you annotate `color_map: dict[str, int]`, a `UserDict` subclass would fail type checking -- even though it works perfectly at runtime. `Mapping` is the safer choice.

---
## 5. TypeVar and Parameterized Generics

See also: [[typevar-and-parameterized-generics]]

`TypeVar` lets you express "the return type depends on the input type." Without it, you lose type information when passing through generic functions.

In [ ]:
from collections.abc import Sequence, Iterable, Hashable
from typing import TypeVar
from random import shuffle

# Basic TypeVar: T can be anything
T = TypeVar("T")

def sample(population: Sequence[T], size: int) -> list[T]:
    "Return `size` random items from population, preserving element type."
    if size < 1:
        raise ValueError("size must be >= 1")
    result = list(population)
    shuffle(result)
    return result[:size]

# If you pass Sequence[int], you get list[int] back
nums = (10, 20, 30, 40, 50)
print(sample(nums, 3))  # list[int]

# If you pass Sequence[str], you get list[str] back
words = ["apple", "banana", "cherry", "date"]
print(sample(words, 2))  # list[str]

### Restricted TypeVar vs Bounded TypeVar

| Feature | Restricted | Bounded |
|---------|-----------|---------|
| Syntax | `TypeVar("T", float, str)` | `TypeVar("T", bound=Hashable)` |
| Meaning | Must be exactly one of the listed types | Must be the bound type or any subtype |
| Use case | Small set of known types | Open-ended subtype hierarchy |

In [ ]:
from collections import Counter
from decimal import Decimal
from fractions import Fraction

# Restricted TypeVar: only these specific types
NumberT = TypeVar("NumberT", float, Decimal, Fraction)

def mode_restricted(data: Iterable[NumberT]) -> NumberT:
    "Return most common value. Only works with float, Decimal, Fraction."
    pairs = Counter(data).most_common(1)
    if len(pairs) == 0:
        raise ValueError("no mode for empty data")
    return pairs[0][0]

print(mode_restricted([1.0, 2.0, 2.0, 3.0]))       # 2.0
print(mode_restricted([Decimal("1"), Decimal("2"), Decimal("2")]))  # 2

# Bounded TypeVar: any Hashable subtype
HashableT = TypeVar("HashableT", bound=Hashable)

def mode_bounded(data: Iterable[HashableT]) -> HashableT:
    "Return most common value. Works with any hashable type."
    pairs = Counter(data).most_common(1)
    if len(pairs) == 0:
        raise ValueError("no mode for empty data")
    return pairs[0][0]

# Now works with str too!
print(mode_bounded(["red", "blue", "blue", "red", "green", "red", "red"]))  # red
print(mode_bounded([1, 1, 2, 3, 3, 3, 3, 4]))  # 3

---
## 6. Static Protocols (Static Duck Typing)

See also: [[static-protocols]]

`typing.Protocol` defines structural interfaces checked by the type checker. A class is consistent-with a Protocol if it implements the required methods -- **no inheritance or registration needed**.

This is the bridge between Python's duck typing and static type checking.

In [ ]:
from typing import Protocol, Any, TypeVar, runtime_checkable

# Define a Protocol: any type with __lt__ is "SupportsLessThan"
@runtime_checkable
class SupportsLessThan(Protocol):
    def __lt__(self, other: Any) -> bool: ...

# Use it as a bound for TypeVar
LT = TypeVar("LT", bound=SupportsLessThan)

def top(series: Iterable[LT], length: int) -> list[LT]:
    "Return the `length` largest items from an iterable."
    ordered = sorted(series, reverse=True)
    return ordered[:length]

# Works with int (has __lt__)
print(top([4, 1, 5, 2, 6, 7, 3], 3))  # [7, 6, 5]

# Works with str (has __lt__)
print(top("mango pear apple kiwi banana".split(), 3))  # ['pear', 'mango', 'kiwi']

# Works with tuple (has __lt__)
fruits = "mango pear apple kiwi banana".split()
pairs = [(len(s), s) for s in fruits]
print(top(pairs, 3))  # [(6, 'banana'), (5, 'mango'), (5, 'apple')]

In [ ]:
# The Protocol is structural -- no need to inherit from it
class Temperature:
    def __init__(self, value: float):
        self.value = value
    def __lt__(self, other: "Temperature") -> bool:
        return self.value < other.value
    def __repr__(self) -> str:
        return f"Temp({self.value})"

# Temperature never mentions SupportsLessThan, but it satisfies the protocol
temps = [Temperature(72), Temperature(65), Temperature(80), Temperature(55)]
print(top(temps, 2))  # [Temp(80), Temp(72)]

# runtime_checkable lets us use isinstance too
print(isinstance(42, SupportsLessThan))          # True
print(isinstance(Temperature(70), SupportsLessThan))  # True
print(isinstance(object(), SupportsLessThan))    # False

---
## 7. Callable, NoReturn, and Variadic Annotations

See also: [[callable-and-noreturn]]

`Callable[[ParamTypes], ReturnType]` annotates function parameters. `NoReturn` marks functions that never return normally.

In [ ]:
from collections.abc import Callable
from typing import NoReturn

# Callable example: a temperature control system
def update(
    probe: Callable[[], float],
    display: Callable[[float], None]
) -> None:
    temperature = probe()
    display(temperature)

def my_probe() -> int:
    "Returns int -- consistent-with float return, so OK as probe."
    return 42

def my_display(temperature: complex) -> None:
    "Accepts complex -- can handle float, so OK as display."
    print(f"Temperature: {temperature}")

update(my_probe, my_display)  # Temperature: 42

# Callable with ... for flexible signatures
def apply_func(func: Callable[..., str], *args: Any, **kwargs: Any) -> str:
    return func(*args, **kwargs)

print(apply_func(str, 42))          # '42'
print(apply_func(", ".join, ["a", "b", "c"]))  # a, b, c

In [ ]:
# NoReturn: for functions that always raise
def never_returns(msg: str) -> NoReturn:
    raise RuntimeError(msg)

try:
    never_returns("Something went wrong!")
except RuntimeError as e:
    print(f"Caught: {e}")

# Variadic parameter annotations
# *args: str means each positional arg is str -> content becomes tuple[str, ...]
# **kwargs: str means each keyword value is str -> attrs becomes dict[str, str]

def tag(
    name: str,
    /,
    *content: str,
    class_: str | None = None,
    **attrs: str,
) -> str:
    attr_parts = []
    if class_:
        attr_parts.append(f' class="{class_}"')
    for key, val in attrs.items():
        attr_parts.append(f' {key}="{val}"')
    attr_str = "".join(attr_parts)
    if content:
        inner = "".join(f"<{name}{attr_str}>{c}</{name}>" for c in content)
        return inner
    return f"<{name}{attr_str} />"

print(tag("br"))
print(tag("p", "hello", "world"))
print(tag("div", "content", class_="main", id="hero"))

---
## Exercises

### Exercise 1: Annotate a Function
Add complete type hints to this function:

In [ ]:
# TODO: Add type hints to parameters and return type
def first_or_default(items, default):
    "Return the first item from a sequence, or default if empty."
    if len(items) == 0:
        return default
    return items[0]

# Test it
print(first_or_default([1, 2, 3], 0))   # 1
print(first_or_default([], "nope"))      # nope

<details>
<summary>Solution</summary>

```python
from typing import TypeVar
from collections.abc import Sequence

T = TypeVar("T")

def first_or_default(items: Sequence[T], default: T) -> T:
    if len(items) == 0:
        return default
    return items[0]
```

Using `TypeVar` ensures the return type matches the element type of the input.
</details>

### Exercise 2: Write a Protocol
Create a `SupportsAdd` Protocol for types that implement `__add__`, then write a `sum_pair` function using it.

In [ ]:
# TODO: Define SupportsAdd Protocol and sum_pair function
# from typing import Protocol, TypeVar

# Your code here...

# Tests (uncomment when done):
# print(sum_pair(3, 4))            # 7
# print(sum_pair("hello ", "world"))  # hello world
# print(sum_pair([1, 2], [3, 4]))    # [1, 2, 3, 4]

<details>
<summary>Solution</summary>

```python
from typing import Protocol, TypeVar, Any

class SupportsAdd(Protocol):
    def __add__(self, other: Any) -> Any: ...

Addable = TypeVar("Addable", bound=SupportsAdd)

def sum_pair(a: Addable, b: Addable) -> Addable:
    return a + b
```
</details>

### Exercise 3: Callable Annotation
Annotate the `apply_twice` function so it takes a callable and a value, applies the callable twice, and returns the result.

In [ ]:
# TODO: Add proper type hints using Callable and TypeVar

def apply_twice(func, value):
    return func(func(value))

# Tests:
print(apply_twice(lambda x: x * 2, 3))     # 12
print(apply_twice(lambda s: s + "!", "hi"))  # hi!!

<details>
<summary>Solution</summary>

```python
from typing import TypeVar
from collections.abc import Callable

T = TypeVar("T")

def apply_twice(func: Callable[[T], T], value: T) -> T:
    return func(func(value))
```

The key insight: `Callable[[T], T]` means the function takes and returns the same type, and `TypeVar` links it to the input value's type.
</details>

---
## Key Takeaways

1. **Gradual typing is opt-in.** You can add hints file by file, function by function. Unannotated code defaults to `Any`.

2. **`Any` vs `object`**: `Any` supports all operations (type checker trusts you). `object` is a real type with minimal operations (type checker enforces).

3. **Postel's Law for type hints**: Accept the most general type that works (`Mapping`, `Iterable`, `Sequence`) in parameters. Return concrete types (`list`, `dict`).

4. **`TypeVar` preserves type relationships** across input and output. Use restricted TypeVars for a small set of types, bounded TypeVars for open-ended hierarchies.

5. **`Protocol` = static duck typing.** Define what operations you need, and any class that implements them qualifies -- no inheritance required.

6. **`Callable` is covariant in return type, contravariant in parameter types.** This matches intuition: a callback returning `int` is fine where `float` is expected, but a callback accepting `int` is NOT fine where one accepting `float` is expected.

7. **Type hints are not a silver bullet.** They catch real bugs in large codebases but cannot express all constraints. Keep writing tests.

---
*Next: Chapter 15 covers type hints for classes, generic classes, variance, and overloaded signatures.*